# Week 1 Day 7 — Custom Dataset: `.csv` / `.npy` / `.pt`
**Jul 7, 2026**

Day 6 built a `Dataset` around tensors already sitting in memory. Real data starts on disk. Today: three `Dataset` subclasses, one per file format, each loading the same underlying data — so you can compare how the file format shapes the loading code, and cross-check that all three produce identical samples.

Scaffold as usual: TODO stubs, hints not formulas, self-check cells.

## Part 1: Write the same data out in three formats

Given. Same synthetic tabular data as Day 6, saved once as `.csv` (via pandas), once as `.npy` (via numpy, features and labels as separate arrays), once as `.pt` (via `torch.save`, a dict of tensors).

In [2]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)
n_samples, n_features = 300, 4

X = torch.randn(n_samples, n_features)
true_w = torch.tensor([0.8, -1.2, 0.5, 1.0])
true_b = -0.2
logits = X @ true_w + true_b + 0.3 * torch.randn(n_samples)
y = (logits > 0).float()

data_dir = Path("day7_data")
data_dir.mkdir(exist_ok=True)

# .csv -- features + label in one table
df = pd.DataFrame(X.numpy(), columns=[f"feature_{i}" for i in range(n_features)])
df["label"] = y.numpy()
df.to_csv(data_dir / "data.csv", index=False)

# .npy -- features and labels as separate raw arrays
np.save(data_dir / "X.npy", X.numpy())
np.save(data_dir / "y.npy", y.numpy())

# .pt -- a dict of tensors, PyTorch's own serialization format
torch.save({"X": X, "y": y}, data_dir / "data.pt")

print("wrote:", list(data_dir.iterdir()))

wrote: [PosixPath('day7_data/data.csv'), PosixPath('day7_data/data.pt'), PosixPath('day7_data/X.npy'), PosixPath('day7_data/y.npy')]


## Part 2: `CSVDataset`

TODO: load `data.csv` with `pandas.read_csv` inside `__init__`, split it into a features tensor and a label tensor (the `label` column vs. everything else), and cache both as attributes — same `__len__`/`__getitem__` pattern as Day 6.

Watch dtype: a pandas DataFrame's `.values` is a numpy array, and by default that may come back as `float64` — worth being deliberate about casting to `float32` when you convert to a tensor, since that's what the rest of these notebooks assume everywhere else.

In [4]:
class CSVDataset(Dataset):
    def __init__(self, csv_path):
        # TODO: read the csv, split into self.X (float32) and self.y (float32)
        df = pd.read_csv(csv_path)
        self.X = torch.tensor(df.iloc[:, :-1].values, dtype=torch.float32)
        self.y = torch.tensor(df.iloc[:, -1].values, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

csv_ds = CSVDataset(data_dir / "data.csv")
assert len(csv_ds) == n_samples
xi, yi = csv_ds[0]
assert xi.dtype == torch.float32 and xi.shape == (n_features,)
print("CSVDataset OK:", len(csv_ds), "samples")

CSVDataset OK: 300 samples


## Part 3: `NpyDataset`

TODO: load `X.npy` and `y.npy` with `np.load`, convert to tensors.

One thing to look into rather than just accept: `np.load` has an `mmap_mode` argument. For a `.npy` file too large to comfortably fit in RAM, what does `mmap_mode="r"` change about *when* the data actually gets read from disk? You don't need to use it here (this file is tiny) — just look it up and note in a comment why it matters at scale.

In [5]:
class NpyDataset(Dataset):
    def __init__(self, x_path, y_path):
        # TODO: load both .npy files, convert to float32 tensors
        self.X = torch.tensor(np.load(x_path), dtype=torch.float32)
        self.y = torch.tensor(np.load(y_path), dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

npy_ds = NpyDataset(data_dir / "X.npy", data_dir / "y.npy")
assert len(npy_ds) == n_samples
print("NpyDataset OK:", len(npy_ds), "samples")

NpyDataset OK: 300 samples


## Part 4: `PtDataset`

TODO: load `data.pt` with `torch.load`, which hands you back the dict exactly as it was saved.

Worth knowing: `torch.load` historically ran on Python's `pickle`, which can execute arbitrary code for maliciously crafted files — a real concern if you ever load a `.pt` from an untrusted source (e.g. a downloaded pretrained checkpoint). Check what `torch.load`'s `weights_only` argument defaults to in your installed torch version, and what it restricts. You don't need to pass it explicitly for this exercise (your own local file is trusted) — just know what the default protects you from.

In [6]:
class PtDataset(Dataset):
    def __init__(self, pt_path):
        # TODO: torch.load the file, pull out X and y
        self.data = torch.load(pt_path)
        self.X = self.data['X']
        self.y = self.data['y']

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

pt_ds = PtDataset(data_dir / "data.pt")
assert len(pt_ds) == n_samples
print("PtDataset OK:", len(pt_ds), "samples")

PtDataset OK: 300 samples


## Part 5: Cross-check all three agree

Given. If Parts 2-4 are correct, every sample should match across all three datasets, since they're three views of the same underlying data.

In [7]:
for i in [0, 1, 100, 299]:
    x_csv, y_csv = csv_ds[i]
    x_npy, y_npy = npy_ds[i]
    x_pt, y_pt = pt_ds[i]
    assert torch.allclose(x_csv, x_npy) and torch.allclose(x_csv, x_pt), f"mismatch at index {i}"
    assert y_csv == y_npy == y_pt, f"label mismatch at index {i}"

print("all three formats agree on every checked sample")

all three formats agree on every checked sample


## Try yourself

1. Time loading + one full `DataLoader` pass over each of the three datasets (`time.perf_counter()` around each). Which format is fastest to *open*? Does that change if you time iterating through all samples afterward?
2. Corrupt one row of `data.csv` by hand (e.g. delete a value so a cell is empty) and see what `CSVDataset` does with it. Then make `__init__` drop or fill missing rows instead of crashing.
3. Rewrite `NpyDataset` to actually use `mmap_mode="r"`, and add a print statement inside `__getitem__` to see when data actually gets touched vs. at construction time.
4. Simulate a dataset too large for RAM: instead of one `.npy` for the whole dataset, save one `.npy` file *per sample* into a directory, and write a `Dataset` that loads a sample's file only inside `__getitem__` (never all at once in `__init__`). This is the pattern real large-scale datasets (e.g. one file per trading day) actually use.